# Startup App Success Analyzer

This notebook runs the project stage by stage and is suitable for an academic demonstration.


## 1. Architecture

Google Play metadata -> description embeddings -> semantic neighbors -> market features -> multiple classifiers -> selected model.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print(ROOT)


## 2. Optional: collect live data

Run this once, then comment it out while experimenting to avoid repeated requests.


In [ ]:
# !python -m src.scrape_playstore --country us --lang en --delay 0.35


## 3. Prepare data and create success target


In [ ]:
from src.prepare_data import prepare
from src.config import RAW_APPS_CSV, PREPARED_CSV

df = prepare(RAW_APPS_CSV, PREPARED_CSV)
df[['title', 'genre', 'realInstalls', 'reviews', 'success_score', 'success']].head()


## 4. Inspect target distribution


In [ ]:
df['success'].value_counts(normalize=True).rename('share')


## 5. Generate semantic embeddings


In [ ]:
import numpy as np
from src.embeddings import encode_descriptions
from src.config import EMBEDDINGS_NPY

embeddings = encode_descriptions(df['description'].tolist())
np.save(EMBEDDINGS_NPY, embeddings)
embeddings.shape


## 6. Build semantic competitor features


In [ ]:
from src.features import build_training_features
from src.config import FEATURES_CSV

features = build_training_features(df, embeddings, top_k=30)
features.to_csv(FEATURES_CSV, index=False)
features.shape


## 7. Train and compare models


In [ ]:
from src.train import train_models
from src.config import MODEL_BUNDLE, METRICS_JSON

bundle, metrics = train_models(FEATURES_CSV, MODEL_BUNDLE, METRICS_JSON)
metrics


## 8. Test a startup idea


In [ ]:
from src.predict import StartupSuccessPredictor, StartupInput

predictor = StartupSuccessPredictor()
result = predictor.predict(StartupInput(
    description='An AI-powered fitness coach that creates personalized workouts, tracks nutrition, adapts routines from progress, and helps users build sustainable exercise habits.',
    summary='AI personal workout and nutrition coach',
    genre='Health & Fitness',
    content_rating='Everyone',
    free=True,
    offers_iap=True,
    contains_ads=False,
), top_k=30)

print('Probability:', f"{result['probability']:.1%}")
result['neighbors'].head(10)
